# 04c. Línea Base Transformers (BETO y RoBERTa)

**Objetivo:** medir desempeño de baselines Transformer sobre el mismo split patient-level.
**Entradas:** `data/splits/train_denoised.csv` y `data/splits/<split>_denoised.csv`.
**Salidas:** `data/beto_eval.csv` y, cuando se ejecutan, `data/roberta_*_eval.csv`, más reportes y predicciones por modelo.
**Notebook anterior:** `notebooks/pipeline/03_denoising_reglas_core.ipynb`.
**Notebook siguiente:** `notebooks/pipeline/06_ingenieria_features_hibridas.ipynb` y `notebooks/pipeline/08_resultados_hibrido_vs_lineas_base.ipynb`.

> **Justificación de continuidad:** BETO se toma como baseline Transformer de referencia y sus embeddings se reutilizan en 06/07 cuando aportan señal útil frente a costo computacional.


## Criterio metodológico
- Este notebook mantiene la lógica baseline de fine-tuning en texto clínico.
- En el corte actual la tarea es binaria (`ansiedad`, `depresion`) con salida probabilística.
- Incluye modo rápido opcional para auditoría técnica (`TRF_MODO_RAPIDO=1`).
- Mantiene un modo de re-evaluación (`TRF_REUSAR_PREDICCIONES=1`) para recalcular métricas sobre `<split>_denoised` sin reentrenar cuando ya existen predicciones.


In [1]:
import os
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from utils_shared import setup_paths, load_splits, calculate_metrics

paths = setup_paths()
DATA_PATH = paths['DATA_PATH']
SPLITS_PATH = paths['SPLITS_PATH']
CHECKPOINTS_PATH = paths['CHECKPOINTS_PATH']
LOGS_PATH = paths['LOGS_PATH']

MODO_RAPIDO = os.getenv('TRF_MODO_RAPIDO', '0') == '1'
EVAL_ON = os.getenv('BASELINE_EVAL_ON', 'dev').strip().lower()
TRF_REUSAR_PREDICCIONES = os.getenv('TRF_REUSAR_PREDICCIONES', '1') == '1'
if EVAL_ON not in {'dev', 'test'}:
    raise ValueError(f"BASELINE_EVAL_ON inválido: {EVAL_ON}. Use 'dev' o 'test'.")
print('MODO_RAPIDO:', MODO_RAPIDO)
print('EVAL_ON:', EVAL_ON)
print('TRF_REUSAR_PREDICCIONES:', TRF_REUSAR_PREDICCIONES)

# Evita inicializar backend de entrenamiento cuando solo se reevalúa
if TRF_REUSAR_PREDICCIONES:
    device = 'reuso_predicciones'
    print('Modo reuso activo: no se inicializan componentes de entrenamiento Transformer.')
else:
    import torch
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print('Usando CUDA')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
        print('Usando MPS')
    else:
        device = torch.device('cpu')
        print('Usando CPU')


/Users/manuelnunez/Projects/psych-phenotyping-paraguay/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MODO_RAPIDO: False
Usando MPS


In [2]:
# Hiperparámetros
MAX_LENGTH = 256 if MODO_RAPIDO else 512
BATCH_SIZE = 2 if MODO_RAPIDO else 4
ACCUMULATION_STEPS = 2 if MODO_RAPIDO else 4
EPOCHS = 1 if MODO_RAPIDO else 3
LEARNING_RATE = 2e-5

MODELOS = {
    'beto': 'dccuchile/bert-base-spanish-wwm-cased',
    'roberta_biomedical': 'PlanTL-GOB-ES/roberta-base-biomedical-es',
    'roberta_clinical': 'PlanTL-GOB-ES/roberta-base-biomedical-clinical-es',
}
if MODO_RAPIDO:
    MODELOS = {'beto': MODELOS['beto']}

print('MAX_LENGTH:', MAX_LENGTH)
print('BATCH_SIZE:', BATCH_SIZE)
print('EPOCHS:', EPOCHS)
print('Modelos:', list(MODELOS.keys()))


MAX_LENGTH: 512
BATCH_SIZE: 4
EPOCHS: 3
Modelos: ['beto', 'roberta_biomedical', 'roberta_clinical']


In [3]:
# Carga de datos
def _load_split_denoised(split_name: str) -> pd.DataFrame:
    p = SPLITS_PATH / f'{split_name}_denoised.csv'
    if not p.exists():
        raise FileNotFoundError(f'No existe {p}. Ejecuta 03_denoising_reglas_core primero.')
    return pd.read_csv(p)

df_train = _load_split_denoised('train')
df_eval = _load_split_denoised(EVAL_ON)

if MODO_RAPIDO:
    df_train = df_train.sample(min(len(df_train), 300), random_state=42)
    df_eval = df_eval.sample(min(len(df_eval), 150), random_state=42)

# Etiquetas detectadas en el corte actual (binaria)
all_labels = sorted(set(df_train['etiqueta'].astype(str)) | set(df_eval['etiqueta'].astype(str)))
label2id = {lab: i for i, lab in enumerate(all_labels)}
id2label = {i: lab for lab, i in label2id.items()}

for df in (df_train, df_eval):
    df['label'] = df['etiqueta'].astype(str).map(label2id)

print('Clases detectadas:', all_labels)
print('Train:', len(df_train), '| Eval:', len(df_eval), f'| split={EVAL_ON}')


Clases detectadas: ['ansiedad', 'depresion']
Train: 1107 | Dev: 595


In [4]:
# Limpieza textual conservadora
RE_MULTI = re.compile(r'(.){2,}')

def clean_text_trf(s: str) -> str:
    if pd.isna(s):
        return ''
    s = str(s).strip()
    s = unicodedata.normalize('NFC', s)
    s = RE_MULTI.sub(r'', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df_train['texto_trf'] = df_train['texto'].map(clean_text_trf)
df_eval['texto_trf'] = df_eval['texto'].map(clean_text_trf)


In [5]:
import inspect

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    pred_ids = np.argmax(predictions, axis=1)
    labels_str = [id2label[int(x)] for x in labels]
    preds_str = [id2label[int(x)] for x in pred_ids]
    m = calculate_metrics(labels_str, preds_str)
    return {
        'f1': m['f1_macro'],
        'precision': m['precision_macro'],
        'recall': m['recall_macro'],
        'accuracy': m['accuracy'],
    }

rows = []
eval_row_ids = set(df_eval['row_id'].astype(int).tolist())

def exportar_metricas_desde_predicciones(model_name: str, pred_df: pd.DataFrame):
    cols_req = {'row_id', 'y_true', 'y_pred'}
    if not cols_req.issubset(pred_df.columns):
        raise ValueError(f'Predicciones de {model_name} sin columnas requeridas: {cols_req}')

    pred_df = pred_df.copy()
    pred_df['row_id'] = pred_df['row_id'].astype(int)
    pred_df = pred_df[pred_df['row_id'].isin(eval_row_ids)].copy()
    if pred_df.empty:
        raise ValueError(f'Predicciones de {model_name} sin intersección con {EVAL_ON}_denoised.')

    true_labels = pred_df['y_true'].astype(str).tolist()
    pred_labels = pred_df['y_pred'].astype(str).tolist()
    m = calculate_metrics(true_labels, pred_labels)

    pred_path = DATA_PATH / f'{model_name}_predicciones_{EVAL_ON}.csv'
    pred_df.to_csv(pred_path, index=False)

    report_path = DATA_PATH / f'{model_name}_classification_report.csv'
    report_df = pd.DataFrame(m['report_dict']).transpose()
    report_df.to_csv(report_path)

    eval_path = DATA_PATH / f'{model_name}_eval.csv'
    metrics_df = pd.DataFrame([{
        'modelo': model_name,
        'f1_macro': m['f1_macro'],
        'precision_macro': m['precision_macro'],
        'recall_macro': m['recall_macro'],
        'accuracy': m['accuracy'],
        'n_train': len(df_train),
        'n_eval': len(pred_df),
        'n_dev': len(pred_df),  # compatibilidad hacia atrás
        'eval_split': EVAL_ON,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'max_length': MAX_LENGTH,
        'n_clases': len(all_labels),
        'clases': '|'.join(all_labels),
        'device': str(device),
    }])
    metrics_df.to_csv(eval_path, index=False)

    rows.append(metrics_df.iloc[0].to_dict())
    print(f'Exportado {eval_path.name}, {report_path.name}, {pred_path.name}')


# El archivo `<modelo>_eval.csv` se utiliza en 08 para comparar líneas base.
for model_name, model_id in MODELOS.items():
    print(f"\n=== Procesando {model_name} ===")

    # Camino rápido: reusar predicciones existentes y recalcular métricas sobre el split denoised
    if TRF_REUSAR_PREDICCIONES:
        pred_candidatos = [
            DATA_PATH / f'{model_name}_predicciones_{EVAL_ON}.csv',
            DATA_PATH / f'{model_name}_predicciones_dev.csv',
        ]
        for p in pred_candidatos:
            if not p.exists():
                continue
            try:
                pred_df = pd.read_csv(p)
                exportar_metricas_desde_predicciones(model_name, pred_df)
                print(f'Reutilizado: {p.name}')
                break
            except Exception as e:
                print(f'Aviso: no se pudo reutilizar {p.name}: {e}')
        else:
            print('No hay predicciones previas reutilizables; se entrena modelo.')
            pred_df = None

        if pred_df is not None and not pred_df.empty:
            continue

    print(f"Entrenando {model_name} desde cero...")

    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

    ds_train = Dataset.from_pandas(df_train[['row_id', 'texto_trf', 'label']].rename(columns={'texto_trf': 'texto'}))
    ds_eval = Dataset.from_pandas(df_eval[['row_id', 'texto_trf', 'label']].rename(columns={'texto_trf': 'texto'}))

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    def tokenize_function(examples):
        return tokenizer(
            examples['texto'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LENGTH,
        )

    tokenized_train = ds_train.map(tokenize_function, batched=True)
    tokenized_eval = ds_eval.map(tokenize_function, batched=True)

    import torch

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=len(all_labels),
        id2label=id2label,
        label2id=label2id,
    )
    model = model.to(device)

    out_ckpt = CHECKPOINTS_PATH / model_name
    out_log = LOGS_PATH / model_name
    out_ckpt.mkdir(parents=True, exist_ok=True)
    out_log.mkdir(parents=True, exist_ok=True)

    ta_params = inspect.signature(TrainingArguments.__init__).parameters
    strategy_key = 'evaluation_strategy' if 'evaluation_strategy' in ta_params else 'eval_strategy'

    ta_kwargs = {
        'output_dir': str(out_ckpt),
        'learning_rate': LEARNING_RATE,
        'per_device_train_batch_size': BATCH_SIZE,
        'per_device_eval_batch_size': BATCH_SIZE,
        'gradient_accumulation_steps': ACCUMULATION_STEPS,
        'gradient_checkpointing': True,
        'num_train_epochs': EPOCHS,
        'weight_decay': 0.01,
        strategy_key: 'epoch',
        'save_strategy': 'epoch',
        'load_best_model_at_end': True,
        'metric_for_best_model': 'f1',
        'logging_dir': str(out_log),
        'logging_steps': 10,
        'seed': 42,
        'report_to': 'none',
    }
    args = TrainingArguments(**ta_kwargs)

    trainer_kwargs = {
        'model': model,
        'args': args,
        'train_dataset': tokenized_train,
        'eval_dataset': tokenized_eval,
        'compute_metrics': compute_metrics,
    }
    trainer_params = inspect.signature(Trainer.__init__).parameters
    if 'tokenizer' in trainer_params:
        trainer_kwargs['tokenizer'] = tokenizer
    else:
        trainer_kwargs['processing_class'] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    trainer.train()
    eval_results = trainer.evaluate()

    # Predicciones para auditoría
    pred_out = trainer.predict(tokenized_eval)
    pred_ids = np.argmax(pred_out.predictions, axis=1)
    pred_labels = [id2label[int(x)] for x in pred_ids]
    true_labels = [id2label[int(x)] for x in pred_out.label_ids]

    pred_df = pd.DataFrame({
        'row_id': ds_eval['row_id'],
        'y_true': true_labels,
        'y_pred': pred_labels,
    })

    # Probabilidades por clase
    probs = torch.nn.functional.softmax(torch.tensor(pred_out.predictions), dim=1).numpy()
    for i, lab in id2label.items():
        pred_df[f'prob_{lab}'] = probs[:, int(i)]

    pred_path = DATA_PATH / f'{model_name}_predicciones_{EVAL_ON}.csv'
    pred_df.to_csv(pred_path, index=False)

    report_path = DATA_PATH / f'{model_name}_classification_report.csv'
    report_df = pd.DataFrame(calculate_metrics(true_labels, pred_labels)['report_dict']).transpose()
    report_df.to_csv(report_path)

    eval_path = DATA_PATH / f'{model_name}_eval.csv'
    metrics_df = pd.DataFrame([{
        'modelo': model_name,
        'f1_macro': eval_results['eval_f1'],
        'precision_macro': eval_results['eval_precision'],
        'recall_macro': eval_results['eval_recall'],
        'accuracy': eval_results['eval_accuracy'],
        'n_train': len(df_train),
        'n_eval': len(df_eval),
        'n_dev': len(df_eval),  # compatibilidad hacia atrás
        'eval_split': EVAL_ON,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'max_length': MAX_LENGTH,
        'n_clases': len(all_labels),
        'clases': '|'.join(all_labels),
        'device': str(device),
    }])
    metrics_df.to_csv(eval_path, index=False)

    rows.append(metrics_df.iloc[0].to_dict())
    print(f'Exportado {eval_path.name}, {report_path.name}, {pred_path.name}')

    del model, trainer, tokenized_train, tokenized_eval
    if hasattr(device, 'type') and device.type == 'cuda':
        torch.cuda.empty_cache()
    elif hasattr(device, 'type') and device.type == 'mps':
        torch.mps.empty_cache()



=== Entrenando beto ===


Map: 100%|██████████| 595/595 [00:00<00:00, 6616.99 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/manuelnunez/Projects/psych-phenotyping-paraguay/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,0.520300,0.597463,0.648111,0.656131,0.643406,0.709244
2,0.405900,0.588017,0.654797,0.662682,0.650005,0.714286
3,0.226100,0.598696,0.657699,0.666631,0.652450,0.717647


/Users/manuelnunez/Projects/psych-phenotyping-paraguay/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/manuelnunez/Projects/psych-phenotyping-paraguay/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/manuelnunez/Projects/psych-phenotyping-paraguay/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


/Users/manuelnunez/Projects/psych-phenotyping-paraguay/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


...

Map: 100%|██████████| 595/595 [00:00<00:00, 6993.58 examples/s]
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at PlanTL-GOB-ES/roberta-base-biomedical-clinical-es and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/manuelnunez/Projects/psych-phenotyping-paraguay/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,0.524700,0.567950,0.645241,0.652288,0.640961,0.705882
2,0.369800,0.554777,0.667021,0.683181,0.659299,0.731092
3,0.264300,0.545488,0.675346,0.692175,0.667120,0.737815


/Users/manuelnunez/Projects/psych-phenotyping-paraguay/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/manuelnunez/Projects/psych-phenotyping-paraguay/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/manuelnunez/Projects/psych-phenotyping-paraguay/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


/Users/manuelnunez/Projects/psych-phenotyping-paraguay/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Exportado roberta_clinical_eval.csv, roberta_clinical_classification_report.csv, roberta_clinical_predicciones_dev.csv


In [6]:
df_resumen = pd.DataFrame(rows)
display(df_resumen.sort_values('f1_macro', ascending=False))
print('Proceso baseline Transformers finalizado.')


,modelo,f1_macro,precision_macro,recall_macro,accuracy,n_train,n_dev,epochs,batch_size,max_length,n_clases,clases,device
2,roberta_clinical,0.675346,0.692175,0.667120,0.737815,1107,595,3,4,512,2,ansiedad|depresion,mps
0,beto,0.657699,0.666631,0.652450,0.717647,1107,595,3,4,512,2,ansiedad|depresion,mps
1,roberta_biomedical,0.657622,0.676272,0.649769,0.726050,1107,595,3,4,512,2,ansiedad|depresion,mps


Proceso baseline Transformers finalizado.
